# Phase 1 — Extraction multi-sources NBA

## Objectif

Cette phase suit le guide sans nettoyer ni transformer prématurément les données.

Nous allons :

1. charger et contrôler le CSV principal ;
2. conserver ce fichier brut sans modification ;
3. tester l'accès à l'API NBA sur une seule saison ;
4. extraire les données par saison avec reprise sur erreur ;
5. sauvegarder chaque réponse en **JSON brut** et en **CSV** ;
6. produire un manifeste des sources et un journal d'extraction ;
7. vérifier les dimensions, types, clés et périodes couvertes.

> La Phase 2 sera consacrée au diagnostic qualité. Ici, on identifie seulement les premiers problèmes sans les corriger.


## Sources retenues

| Source | Endpoint / fichier | Apport principal | Clé attendue |
|---|---|---|---|
| Kaggle | `all_seasons.csv` | Source initiale et référence | nom + saison, provisoire |
| NBA API | `LeagueDashPlayerBioStats` | `PLAYER_ID`, `TEAM_ID`, profil et stats avancées | `PLAYER_ID + SEASON` |
| NBA API | `LeagueDashPlayerStats` — Base | minutes, tirs, titularisation indirecte, turnovers | `PLAYER_ID + SEASON` |
| NBA API | `LeagueDashPlayerStats` — Advanced | rating, pace, usage et efficacité | `PLAYER_ID + SEASON` |
| NBA API | `LeagueDashTeamStats` — Base | résultats et statistiques d'équipe | `TEAM_ID + SEASON` |
| NBA API | `LeagueDashTeamStats` — Advanced | pace et ratings d'équipe | `TEAM_ID + SEASON` |
| NBA API | `CommonTeamRoster` | position et roster | `PLAYER_ID + TEAM_ID + SEASON` |

L'extraction des rosters est optionnelle et désactivée par défaut car elle nécessite un appel par équipe et par saison.


In [ ]:
# À exécuter une seule fois dans l'environnement du projet.
# Décommentez la ligne suivante si les dépendances ne sont pas installées.
#%pip install pandas numpy pyarrow nba_api tqdm



   ---------------------------------------- 0/2 [tqdm]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   -------------------- ------------------- 1/2 [nba_api]
   ---------------------------------------- 2/2 [nba_api]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable
import json
import random
import time
import traceback

import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "ML_NBA_Project" / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT / "ML_NBA_Project"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
API_RAW_DIR = RAW_DIR / "api"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
LOG_DIR = PROJECT_ROOT / "logs"

for directory in (RAW_DIR, API_RAW_DIR, INTERIM_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", PROJECT_ROOT.resolve())


Racine du projet : C:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model


## 1. Chargement du CSV principal

Le fichier brut est uniquement lu et audité. La colonne d'index inutile ou les incohérences éventuelles ne seront traitées qu'en Phase 3.


In [6]:
CSV_CANDIDATES = [
    RAW_DIR / "kaggle" / "all_seasons.csv",
    PROJECT_ROOT / "all_seasons.csv",
    PROJECT_ROOT / "all_seasons(2).csv",
]

csv_path = next((p for p in CSV_CANDIDATES if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "Placez all_seasons.csv dans data/raw/kaggle/ puis relancez cette cellule."
    )

df_kaggle_raw = pd.read_csv(csv_path)

print("Fichier :", csv_path)
print("Dimensions :", df_kaggle_raw.shape)
display(df_kaggle_raw.head())
df_kaggle_raw.info()


Fichier : c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\all_seasons.csv
Dimensions : (12844, 22)


,Unnamed: 0,player_name,team_abbreviation,age,player_height,player_weight,college,country,draft_year,draft_round,draft_number,gp,pts,reb,ast,net_rating,oreb_pct,dreb_pct,usg_pct,ts_pct,ast_pct,season
0,0,Randy Livingston,HOU,22.0,193.04,94.800728,Louisiana State,USA,1996,2,42,64,3.9,1.5,2.4,0.3,0.042,0.071,0.169,0.487,0.248,1996-97
1,1,Gaylon Nickerson,WAS,28.0,190.50,86.182480,Northwestern Oklahoma,USA,1994,2,34,4,3.8,1.3,0.3,8.9,0.030,0.111,0.174,0.497,0.043,1996-97
2,2,George Lynch,VAN,26.0,203.20,103.418976,North Carolina,USA,1993,1,12,41,8.3,6.4,1.9,-8.2,0.106,0.185,0.175,0.512,0.125,1996-97
3,3,George McCloud,LAL,30.0,203.20,102.058200,Florida State,USA,1989,1,7,64,10.2,2.8,1.7,-2.7,0.027,0.111,0.206,0.527,0.125,1996-97
4,4,George Zidek,DEN,23.0,213.36,119.748288,UCLA,USA,1995,1,22,52,2.8,1.7,0.3,-14.1,0.102,0.169,0.195,0.500,0.064,1996-97


<class 'pandas.DataFrame'>
RangeIndex: 12844 entries, 0 to 12843
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         12844 non-null  int64  
 1   player_name        12844 non-null  str    
 2   team_abbreviation  12844 non-null  str    
 3   age                12844 non-null  float64
 4   player_height      12844 non-null  float64
 5   player_weight      12844 non-null  float64
 6   college            10990 non-null  str    
 7   country            12844 non-null  str    
 8   draft_year         12844 non-null  str    
 9   draft_round        12844 non-null  str    
 10  draft_number       12844 non-null  str    
 11  gp                 12844 non-null  int64  
 12  pts                12844 non-null  float64
 13  reb                12844 non-null  float64
 14  ast                12844 non-null  float64
 15  net_rating         12844 non-null  float64
 16  oreb_pct           12844 non-null

In [7]:
# Premiers contrôles d'extraction, sans nettoyage.
source_kaggle_summary = pd.DataFrame([{
    "source": "Kaggle - NBA Players",
    "format_original": "CSV",
    "fichier_endpoint": str(csv_path.relative_to(PROJECT_ROOT)),
    "lignes": len(df_kaggle_raw),
    "colonnes": df_kaggle_raw.shape[1],
    "periode_min": df_kaggle_raw["season"].min(),
    "periode_max": df_kaggle_raw["season"].max(),
    "cle_candidate": "player_name + season",
    "observations": (
        "Chargement réussi ; colonne d'index Unnamed: 0 ; "
        "PLAYER_ID absent ; clé provisoire non fiable pour les homonymes."
    ),
}])

display(source_kaggle_summary)


,source,format_original,fichier_endpoint,lignes,colonnes,periode_min,periode_max,cle_candidate,observations
0,Kaggle - NBA Players,CSV,all_seasons.csv,12844,22,1996-97,2022-23,player_name + season,Chargement réussi ; colonne d'index Unnamed: 0...


## 2. Configuration de l'API NBA

Le package `nba_api` envoie les requêtes à `stats.nba.com`. Les appels peuvent parfois expirer. Le code ci-dessous :

- utilise un délai entre les requêtes ;
- réessaie avec attente progressive ;
- conserve les fichiers déjà téléchargés ;
- journalise chaque succès ou échec ;
- permet de relancer le notebook sans recommencer tout l'historique.

La première exécution doit se limiter à `SMOKE_TEST_SEASON`.


In [21]:
try:
    from nba_api.stats.endpoints import (
        leaguedashplayerbiostats,
        leaguedashplayerstats,
        leaguedashteamstats,
        commonteamroster,
    )
except ImportError as exc:
    raise ImportError(
        "Le package nba_api n'est pas installé. "
        "Exécutez `%pip install -r requirements.txt`, puis redémarrez le kernel."
    ) from exc

START_SEASON_YEAR = 1996
LAST_COMPLETE_SEASON_YEAR = 2022  # Produit la saison 2025-26.
SMOKE_TEST_SEASON = "2022-23"

# Commencer par False. Passer à True seulement après le test d'une saison.
FETCH_FULL_HISTORY = True

# Les rosters demandent environ 30 appels supplémentaires par saison.
FETCH_ROSTERS = True

REQUEST_TIMEOUT_SECONDS = 75
MAX_ATTEMPTS = 5
SLEEP_BETWEEN_REQUESTS = (1.5, 3.0)

def season_label(start_year: int) -> str:
    return f"{start_year}-{str(start_year + 1)[-2:]}"

ALL_SEASONS = [
    season_label(year)
    for year in range(START_SEASON_YEAR, LAST_COMPLETE_SEASON_YEAR + 1)
]

SEASONS_TO_FETCH = ALL_SEASONS if FETCH_FULL_HISTORY else [SMOKE_TEST_SEASON]

print("Saisons prévues :", SEASONS_TO_FETCH[:3], "...", SEASONS_TO_FETCH[-3:])
print("Nombre de saisons :", len(SEASONS_TO_FETCH))


Saisons prévues : ['1996-97', '1997-98', '1998-99'] ... ['2020-21', '2021-22', '2022-23']
Nombre de saisons : 27


In [14]:
@dataclass
class ExtractionRecord:
    timestamp_utc: str
    source: str
    season: str
    status: str
    rows: int | None
    columns: int | None
    json_path: str | None
    csv_path: str | None
    attempt: int
    error: str | None

EXTRACTION_RECORDS: list[ExtractionRecord] = []

def _safe_season_name(season: str) -> str:
    return season.replace("-", "_")

def _append_log(record: ExtractionRecord) -> None:
    EXTRACTION_RECORDS.append(record)
    log_path = LOG_DIR / "extraction_log.csv"
    pd.DataFrame([asdict(r) for r in EXTRACTION_RECORDS]).to_csv(
        log_path, index=False, encoding="utf-8"
    )

def fetch_and_save(
    source_name: str,
    season: str,
    endpoint_factory: Callable[[], object],
    dataset_index: int = 0,
    force: bool = False,
) -> pd.DataFrame:
    source_dir = API_RAW_DIR / source_name
    source_dir.mkdir(parents=True, exist_ok=True)

    suffix = _safe_season_name(season)
    json_path = source_dir / f"{source_name}_{suffix}.json"
    csv_path = source_dir / f"{source_name}_{suffix}.csv"

    if csv_path.exists() and json_path.exists() and not force:
        df_cached = pd.read_csv(csv_path)
        print(f"[CACHE] {source_name} {season}: {len(df_cached)} lignes")
        return df_cached

    last_error: Exception | None = None

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            endpoint = endpoint_factory()
            frames = endpoint.get_data_frames()
            if len(frames) <= dataset_index:
                raise RuntimeError(
                    f"Dataset index {dataset_index} absent. "
                    f"Nombre de DataFrames reçus : {len(frames)}"
                )

            df = frames[dataset_index].copy()
            if df.empty:
                raise RuntimeError("Réponse vide reçue depuis l'API.")

            if "SEASON" not in df.columns and "SEASON_YEAR" not in df.columns:
                df["SEASON"] = season

            # JSON brut fourni par le client, conservé tel quel.
            json_path.write_text(endpoint.get_json(), encoding="utf-8")
            df.to_csv(csv_path, index=False, encoding="utf-8")

            record = ExtractionRecord(
                timestamp_utc=datetime.now(timezone.utc).isoformat(),
                source=source_name,
                season=season,
                status="success",
                rows=len(df),
                columns=df.shape[1],
                json_path=str(json_path.relative_to(PROJECT_ROOT)),
                csv_path=str(csv_path.relative_to(PROJECT_ROOT)),
                attempt=attempt,
                error=None,
            )
            _append_log(record)
            print(f"[OK] {source_name} {season}: {df.shape}")
            return df

        except Exception as exc:
            last_error = exc
            error_text = f"{type(exc).__name__}: {exc}"
            print(f"[ERREUR] {source_name} {season}, tentative {attempt}: {error_text}")

            if attempt < MAX_ATTEMPTS:
                wait_seconds = min(60, (2 ** (attempt - 1)) * 3 + random.uniform(0, 2))
                print(f"Nouvelle tentative dans {wait_seconds:.1f} s")
                time.sleep(wait_seconds)

    record = ExtractionRecord(
        timestamp_utc=datetime.now(timezone.utc).isoformat(),
        source=source_name,
        season=season,
        status="failed",
        rows=None,
        columns=None,
        json_path=None,
        csv_path=None,
        attempt=MAX_ATTEMPTS,
        error=f"{type(last_error).__name__}: {last_error}" if last_error else "Erreur inconnue",
    )
    _append_log(record)
    raise RuntimeError(
        f"Échec définitif pour {source_name}, saison {season}."
    ) from last_error


## 3. Fonctions d'extraction par source

Les paramètres sont explicitement fixés pour limiter les changements silencieux de comportement de l'API.


In [15]:
def fetch_player_bio(season: str, force: bool = False) -> pd.DataFrame:
    return fetch_and_save(
        source_name="player_bio",
        season=season,
        endpoint_factory=lambda: leaguedashplayerbiostats.LeagueDashPlayerBioStats(
            league_id="00",
            per_mode_simple="PerGame",
            season=season,
            season_type_all_star="Regular Season",
            timeout=REQUEST_TIMEOUT_SECONDS,
        ),
        force=force,
    )

def fetch_player_stats_base(season: str, force: bool = False) -> pd.DataFrame:
    return fetch_and_save(
        source_name="player_stats_base",
        season=season,
        endpoint_factory=lambda: leaguedashplayerstats.LeagueDashPlayerStats(
            league_id_nullable="00",
            measure_type_detailed_defense="Base",
            per_mode_detailed="PerGame",
            season=season,
            season_type_all_star="Regular Season",
            timeout=REQUEST_TIMEOUT_SECONDS,
        ),
        force=force,
    )

def fetch_player_stats_advanced(season: str, force: bool = False) -> pd.DataFrame:
    return fetch_and_save(
        source_name="player_stats_advanced",
        season=season,
        endpoint_factory=lambda: leaguedashplayerstats.LeagueDashPlayerStats(
            league_id_nullable="00",
            measure_type_detailed_defense="Advanced",
            per_mode_detailed="PerGame",
            season=season,
            season_type_all_star="Regular Season",
            timeout=REQUEST_TIMEOUT_SECONDS,
        ),
        force=force,
    )

def fetch_team_stats_base(season: str, force: bool = False) -> pd.DataFrame:
    return fetch_and_save(
        source_name="team_stats_base",
        season=season,
        endpoint_factory=lambda: leaguedashteamstats.LeagueDashTeamStats(
            league_id_nullable="00",
            measure_type_detailed_defense="Base",
            per_mode_detailed="PerGame",
            season=season,
            season_type_all_star="Regular Season",
            timeout=REQUEST_TIMEOUT_SECONDS,
        ),
        force=force,
    )

def fetch_team_stats_advanced(season: str, force: bool = False) -> pd.DataFrame:
    return fetch_and_save(
        source_name="team_stats_advanced",
        season=season,
        endpoint_factory=lambda: leaguedashteamstats.LeagueDashTeamStats(
            league_id_nullable="00",
            measure_type_detailed_defense="Advanced",
            per_mode_detailed="PerGame",
            season=season,
            season_type_all_star="Regular Season",
            timeout=REQUEST_TIMEOUT_SECONDS,
        ),
        force=force,
    )


## 4. Test obligatoire sur une saison

Exécuter cette cellule avant l'historique complet. Vérifier que chaque tableau contient des lignes, les identifiants attendus et la bonne saison.


In [16]:
test_season = SMOKE_TEST_SEASON

smoke_results = {
    "player_bio": fetch_player_bio(test_season),
    "player_stats_base": fetch_player_stats_base(test_season),
    "player_stats_advanced": fetch_player_stats_advanced(test_season),
    "team_stats_base": fetch_team_stats_base(test_season),
    "team_stats_advanced": fetch_team_stats_advanced(test_season),
}

smoke_summary = pd.DataFrame([
    {
        "source": name,
        "season": test_season,
        "rows": len(frame),
        "columns": frame.shape[1],
        "contains_player_id": "PLAYER_ID" in frame.columns,
        "contains_team_id": "TEAM_ID" in frame.columns,
    }
    for name, frame in smoke_results.items()
])

display(smoke_summary)

for name, frame in smoke_results.items():
    print("\n", "=" * 30, name, "=" * 30)
    display(frame.head(3))


[CACHE] player_bio 2022-23: 539 lignes
[CACHE] player_stats_base 2022-23: 539 lignes
[CACHE] player_stats_advanced 2022-23: 539 lignes
[CACHE] team_stats_base 2022-23: 30 lignes
[CACHE] team_stats_advanced 2022-23: 30 lignes


,source,season,rows,columns,contains_player_id,contains_team_id
0,player_bio,2022-23,539,24,True,True
1,player_stats_base,2022-23,539,68,True,True
2,player_stats_advanced,2022-23,539,80,True,True
3,team_stats_base,2022-23,30,55,False,True
4,team_stats_advanced,2022-23,30,47,False,True



 ============================== player_bio ==============================


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,AGE,PLAYER_HEIGHT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,COLLEGE,COUNTRY,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER,GP,PTS,REB,AST,NET_RATING,OREB_PCT,DREB_PCT,USG_PCT,TS_PCT,AST_PCT,SEASON
0,1630639,A.J. Lawson,1610612742,DAL,22.0,6-6,78,179,South Carolina,Canada,Undrafted,Undrafted,Undrafted,15,3.7,1.4,0.1,-20.1,0.046,0.152,0.189,0.589,0.032,2022-23
1,1631260,AJ Green,1610612749,MIL,23.0,6-5,77,190,Northern Iowa,USA,Undrafted,Undrafted,Undrafted,35,4.4,1.3,0.6,-4.9,0.016,0.105,0.159,0.607,0.092,2022-23
2,1631100,AJ Griffin,1610612737,ATL,19.0,6-6,78,220,Duke,USA,2022,1,16,72,8.9,2.1,1.0,1.5,0.026,0.080,0.174,0.577,0.070,2022-23



 ============================== player_stats_base ==============================


...

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM,FGA,FGM_PG,FGA_PG,FG_PCT,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,E_OFF_RATING_RANK,OFF_RATING_RANK,sp_work_OFF_RATING_RANK,E_DEF_RATING_RANK,DEF_RATING_RANK,sp_work_DEF_RATING_RANK,E_NET_RATING_RANK,NET_RATING_RANK,sp_work_NET_RATING_RANK,AST_PCT_RANK,AST_TO_RANK,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,E_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,USG_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK,PACE_RANK,sp_work_PACE_RANK,PIE_RANK,FGM_RANK,FGA_RANK,FGM_PG_RANK,FGA_PG_RANK,FG_PCT_RANK,TEAM_COUNT,SEASON
0,1630639,A.J. Lawson,A.J.,1610612742,DAL,22.0,15,5,10,0.333,7.2,99.3,96.3,96.3,116.9,116.4,116.4,-17.7,-20.1,-20.1,0.032,0.67,3.8,0.046,0.152,0.091,5.8,5.7,0.614,0.589,0.189,0.202,104.09,105.96,88.30,105.96,0.076,240,22,44,1.5,2.9,0.500,451,459,105,442,484,510,514,514,479,455,455,521,524,524,519,483,522,203,164,207,47,46,71,203,188,163,87,35,35,327,438,445,417,438,140,2,2022-23
1,1631260,AJ Green,AJ,1610612749,MIL,23.0,35,27,8,0.771,9.9,110.1,107.9,107.9,113.9,112.8,112.8,-3.8,-4.9,-4.9,0.092,2.44,14.0,0.016,0.105,0.061,5.7,5.7,0.600,0.607,0.159,0.167,101.40,102.90,85.75,102.90,0.076,744,53,125,1.5,3.6,0.424,379,238,87,14,448,297,384,384,371,300,300,358,378,378,332,117,348,431,340,394,44,44,93,142,325,312,303,159,159,330,390,376,411,386,378,1,2022-23
2,1631100,AJ Griffin,AJ,1610612737,ATL,19.0,72,34,38,0.472,19.5,113.9,113.6,113.6,111.6,112.0,112.0,2.3,1.5,1.5,0.070,1.74,10.9,0.026,0.080,0.053,6.3,6.3,0.560,0.577,0.174,0.180,102.72,102.67,85.56,102.67,0.077,3005,248,533,3.4,7.4,0.465,102,149,459,326,260,129,169,169,262,262,262,166,196,196,421,252,427,338,467,450,65,64,194,241,260,252,194,178,178,320,170,160,208,199,240,1,2022-23



 ============================== team_stats_base ==============================


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,SEASON
0,1610612737,Atlanta Hawks,82,41,41,0.500,48.4,44.6,92.4,0.483,10.8,30.5,0.352,18.5,22.6,0.818,11.2,33.2,44.4,25.0,12.9,7.1,4.9,5.0,18.8,19.7,118.4,0.3,1,17,17,17,8,1,3,9,26,28,21,16,20,3,9,12,10,18,3,18,12,21,6,19,3,18,2022-23
1,1610612738,Boston Celtics,82,57,25,0.695,48.7,42.2,88.8,0.475,16.0,42.6,0.377,17.5,21.6,0.812,9.7,35.6,45.3,26.7,13.4,6.4,5.2,3.9,18.8,19.1,117.9,6.5,1,2,2,2,1,13,15,14,2,2,6,24,28,4,20,3,7,7,7,27,6,4,7,25,4,1,2022-23
2,1610612751,Brooklyn Nets,82,45,37,0.549,48.1,41.5,85.1,0.487,12.8,33.8,0.378,17.7,22.1,0.800,8.2,32.3,40.5,25.5,13.7,7.1,6.2,3.9,21.1,18.5,113.4,0.9,1,9,9,9,29,21,28,7,9,13,5,19,25,7,29,20,29,13,14,16,1,3,25,28,20,15,2022-23



 ============================== team_stats_advanced ==============================


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,E_DEF_RATING,DEF_RATING,E_NET_RATING,NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,EFG_PCT,TS_PCT,E_PACE,PACE,PACE_PER40,POSS,PIE,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,OFF_RATING_RANK,DEF_RATING_RANK,NET_RATING_RANK,AST_PCT_RANK,AST_TO_RANK,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SEASON
0,1610612737,Atlanta Hawks,82,41,41,0.500,3971.0,113.9,115.5,113.6,115.4,0.3,0.1,0.560,1.93,17.6,0.289,0.717,0.500,0.126,0.541,0.579,103.1,101.56,84.63,8407,0.500,1,17,17,17,8,7,22,19,28,6,22,11,18,16,3,19,19,6,17,2022-23
1,1610612738,Boston Celtics,82,57,25,0.695,3996.0,115.7,117.3,108.8,110.6,7.0,6.7,0.632,2.00,19.1,0.255,0.746,0.506,0.133,0.566,0.600,100.6,99.15,82.63,8247,0.534,1,2,2,2,1,2,2,1,5,4,5,27,1,9,7,4,5,20,1,2022-23
2,1610612751,Brooklyn Nets,82,45,37,0.549,3946.0,113.0,114.6,110.9,113.5,2.1,1.1,0.616,1.86,18.9,0.234,0.689,0.472,0.139,0.562,0.598,100.7,98.77,82.31,8111,0.510,1,9,9,9,29,12,16,14,9,11,6,29,29,29,16,7,7,23,13,2022-23


In [17]:
# Contrôles minimaux du test.
assert not smoke_results["player_bio"].empty
assert "PLAYER_ID" in smoke_results["player_bio"].columns
assert "TEAM_ID" in smoke_results["player_bio"].columns

assert "PLAYER_ID" in smoke_results["player_stats_base"].columns
assert "MIN" in smoke_results["player_stats_base"].columns
assert "FGA" in smoke_results["player_stats_base"].columns

assert "PLAYER_ID" in smoke_results["player_stats_advanced"].columns
assert "TEAM_ID" in smoke_results["team_stats_base"].columns
assert "TEAM_ID" in smoke_results["team_stats_advanced"].columns

print("Test d'extraction validé pour", test_season)


Test d'extraction validé pour 2022-23


## 5. Extraction historique

Ne lancer cette cellule qu'après avoir :

1. validé le test précédent ;
2. placé `FETCH_FULL_HISTORY = True` dans la configuration ;
3. relancé la cellule de configuration.

Chaque source est sauvegardée saison par saison. Une interruption n'oblige pas à recommencer les fichiers déjà présents.


In [18]:
if not FETCH_FULL_HISTORY:
    print(
        "Historique complet désactivé. "
        "Passez FETCH_FULL_HISTORY à True après validation du test."
    )
else:
    for season in tqdm(SEASONS_TO_FETCH, desc="Saisons"):
        fetch_player_bio(season)
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS))

        fetch_player_stats_base(season)
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS))

        fetch_player_stats_advanced(season)
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS))

        fetch_team_stats_base(season)
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS))

        fetch_team_stats_advanced(season)
        time.sleep(random.uniform(*SLEEP_BETWEEN_REQUESTS))


Saisons:   0%|          | 0/27 [00:00<?, ?it/s]

[OK] player_bio 1996-97: (441, 24)
[OK] player_stats_base 1996-97: (441, 68)
[OK] player_stats_advanced 1996-97: (441, 80)
[OK] team_stats_base 1996-97: (29, 55)
[OK] team_stats_advanced 1996-97: (29, 47)
[OK] player_bio 1997-98: (439, 24)
[OK] player_stats_base 1997-98: (439, 68)
[OK] player_stats_advanced 1997-98: (439, 80)
[OK] team_stats_base 1997-98: (29, 55)
[OK] team_stats_advanced 1997-98: (29, 47)
[OK] player_bio 1998-99: (440, 24)
[OK] player_stats_base 1998-99: (440, 68)
[OK] player_stats_advanced 1998-99: (440, 80)
[OK] team_stats_base 1998-99: (29, 55)
[OK] team_stats_advanced 1998-99: (29, 47)
[OK] player_bio 1999-00: (439, 24)
[OK] player_stats_base 1999-00: (439, 68)
[OK] player_stats_advanced 1999-00: (439, 80)
[OK] team_stats_base 1999-00: (29, 55)
[OK] team_stats_advanced 1999-00: (29, 47)
[OK] player_bio 2000-01: (441, 24)
[OK] player_stats_base 2000-01: (441, 68)
[OK] player_stats_advanced 2000-01: (441, 80)
[OK] team_stats_base 2000-01: (29, 55)
[OK] team_stats_ad

## 6. Extraction optionnelle des rosters et positions

Cette partie utilise les `TEAM_ID` déjà extraits par `team_stats_base`. Elle est volontairement séparée car elle peut produire plusieurs centaines d'appels.

Conseil : commencer sur les dernières saisons seulement, puis élargir si les appels sont stables.


In [ ]:
def fetch_roster(season: str, team_id: int, force: bool = False) -> pd.DataFrame:
    source_name = "rosters"
    source_dir = API_RAW_DIR / source_name
    source_dir.mkdir(parents=True, exist_ok=True)

    suffix = _safe_season_name(season)
    json_path = source_dir / f"roster_{team_id}_{suffix}.json"
    csv_path = source_dir / f"roster_{team_id}_{suffix}.csv"

    if csv_path.exists() and json_path.exists() and not force:
        return pd.read_csv(csv_path)

    last_error: Exception | None = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            endpoint = commonteamroster.CommonTeamRoster(
                team_id=team_id,
                season=season,
                league_id_nullable="00",
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            df = endpoint.get_data_frames()[0].copy()
            if df.empty:
                raise RuntimeError("Roster vide.")

            df["SEASON"] = season
            json_path.write_text(endpoint.get_json(), encoding="utf-8")
            df.to_csv(csv_path, index=False, encoding="utf-8")

            _append_log(ExtractionRecord(
                timestamp_utc=datetime.now(timezone.utc).isoformat(),
                source=source_name,
                season=season,
                status="success",
                rows=len(df),
                columns=df.shape[1],
                json_path=str(json_path.relative_to(PROJECT_ROOT)),
                csv_path=str(csv_path.relative_to(PROJECT_ROOT)),
                attempt=attempt,
                error=None,
            ))
            return df
        except Exception as exc:
            last_error = exc
            if attempt < MAX_ATTEMPTS:
                time.sleep(min(60, 3 * 2 ** (attempt - 1) + random.uniform(0, 2)))

    raise RuntimeError(
        f"Échec roster équipe {team_id}, saison {season}"
    ) from last_error


if not FETCH_ROSTERS:
    print("Extraction des rosters désactivée.")
else:
    roster_seasons = ALL_SEASONS

    for season in tqdm(roster_seasons, desc="Saisons rosters"):
        team_file = (
            API_RAW_DIR
            / "team_stats_base"
            / f"team_stats_base_{_safe_season_name(season)}.csv"
        )
        if not team_file.exists():
            raise FileNotFoundError(
                f"Statistiques équipes absentes pour {season}: {team_file}"
            )

        team_ids = (
            pd.read_csv(team_file)["TEAM_ID"]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        for team_id in tqdm(team_ids, desc=season, leave=False):
            fetch_roster(season, team_id)
            time.sleep(random.uniform(2.0, 4.0))


Saisons rosters:   0%|          | 0/27 [00:00<?, ?it/s]

1996-97:   0%|          | 0/29 [00:00<?, ?it/s]

1997-98:   0%|          | 0/29 [00:00<?, ?it/s]

1998-99:   0%|          | 0/29 [00:00<?, ?it/s]

1999-00:   0%|          | 0/29 [00:00<?, ?it/s]

...

2018-19:   0%|          | 0/30 [00:00<?, ?it/s]

2019-20:   0%|          | 0/30 [00:00<?, ?it/s]

2020-21:   0%|          | 0/30 [00:00<?, ?it/s]

2021-22:   0%|          | 0/30 [00:00<?, ?it/s]

2022-23:   0%|          | 0/30 [00:00<?, ?it/s]

## 7. Consolidation des fichiers par source

Cette étape concatène les fichiers saisonniers sans encore réaliser de jointure entre les sources. Les tables consolidées restent des résultats d'extraction.


In [24]:
def prepare_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Harmonise les colonnes contenant des types mixtes pour permettre
    l'enregistrement en Parquet.

    Les colonnes object sont conservées sous forme de texte.
    Leur conversion définitive sera faite pendant le nettoyage.
    """
    df_safe = df.copy()

    for column in df_safe.select_dtypes(include=["object"]).columns:
        df_safe[column] = df_safe[column].astype("string")

    return df_safe


def consolidate_source(
    source_name: str,
    file_pattern: str | None = None
) -> pd.DataFrame:

    source_dir = API_RAW_DIR / source_name

    if file_pattern is None:
        file_pattern = f"{source_name}_*.csv"

    files = sorted(source_dir.glob(file_pattern))

    if not files:
        print(
            f"Aucun fichier à consolider pour {source_name} "
            f"avec le motif {file_pattern}"
        )
        return pd.DataFrame()

    frames = []

    for file in files:
        frame = pd.read_csv(
            file,
            low_memory=False
        )

        frame["SOURCE_FILE"] = file.name
        frames.append(frame)

    combined = pd.concat(
        frames,
        ignore_index=True,
        sort=False
    )

    # Harmonisation technique pour l'écriture Parquet
    combined_safe = prepare_for_parquet(combined)

    output_path = (
        INTERIM_DIR
        / f"{source_name}_all_seasons.parquet"
    )

    combined_safe.to_parquet(
        output_path,
        index=False,
        engine="pyarrow"
    )

    print(
        f"{source_name} : "
        f"{len(files)} fichiers, "
        f"{combined_safe.shape} -> {output_path}"
    )

    return combined_safe


sources_to_consolidate = {
    "player_bio": "player_bio_*.csv",
    "player_stats_base": "player_stats_base_*.csv",
    "player_stats_advanced": "player_stats_advanced_*.csv",
    "team_stats_base": "team_stats_base_*.csv",
    "team_stats_advanced": "team_stats_advanced_*.csv",
    "rosters": "roster_*.csv",
}


consolidated = {}

for source_name, file_pattern in sources_to_consolidate.items():
    consolidated[source_name] = consolidate_source(
        source_name=source_name,
        file_pattern=file_pattern
    )

C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_safe.select_dtypes(include=["object"]).columns:


player_bio : 27 fichiers, (12846, 25) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\player_bio_all_seasons.parquet


C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_safe.select_dtypes(include=["object"]).columns:


player_stats_base : 27 fichiers, (12846, 69) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\player_stats_base_all_seasons.parquet


C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_safe.select_dtypes(include=["object"]).columns:


player_stats_advanced : 27 fichiers, (12846, 81) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\player_stats_advanced_all_seasons.parquet
team_stats_base : 27 fichiers, (802, 56) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\team_stats_base_all_seasons.parquet


C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_safe.select_dtypes(include=["object"]).columns:
C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/u

team_stats_advanced : 27 fichiers, (802, 48) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\team_stats_advanced_all_seasons.parquet
rosters : 802 fichiers, (11672, 18) -> c:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\rosters_all_seasons.parquet


C:\Users\axeln\AppData\Local\Temp\ipykernel_12308\681897539.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_safe.select_dtypes(include=["object"]).columns:


## 8. Manifeste des sources

Le manifeste correspond au tableau récapitulatif demandé dans le guide : origine, format, volume, période, clé candidate et premiers problèmes observés.


In [ ]:
manifest_rows = source_kaggle_summary.to_dict("records")


# Clés candidates prévues pour chaque source
source_keys = {
    "player_bio": "PLAYER_ID + SEASON",
    "player_stats_base": "PLAYER_ID + SEASON",
    "player_stats_advanced": "PLAYER_ID + SEASON",
    "team_stats_base": "TEAM_ID + SEASON",
    "team_stats_advanced": "TEAM_ID + SEASON",

    # Un joueur peut apparaître dans plusieurs équipes durant une saison
    "rosters": "PLAYER_ID + TEAM_ID + SEASON",
}


# Nom réel ou description de chaque endpoint
source_endpoints = {
    "player_bio": "LeagueDashPlayerBioStats",
    "player_stats_base": "LeagueDashPlayerStats - Base",
    "player_stats_advanced": "LeagueDashPlayerStats - Advanced",
    "team_stats_base": "LeagueDashTeamStats - Base",
    "team_stats_advanced": "LeagueDashTeamStats - Advanced",
    "rosters": "CommonTeamRoster",
}


# Description spécifique de chaque source
source_observations = {
    "player_bio": (
        "Profils joueurs et identifiants NBA. "
        "JSON brut et CSV saisonniers conservés."
    ),
    "player_stats_base": (
        "Statistiques traditionnelles par joueur et par saison : "
        "minutes, tirs, points, rebonds, passes et turnovers."
    ),
    "player_stats_advanced": (
        "Statistiques avancées par joueur : usage, ratings, pace "
        "et indicateurs d'efficacité."
    ),
    "team_stats_base": (
        "Statistiques traditionnelles des équipes par saison."
    ),
    "team_stats_advanced": (
        "Statistiques avancées des équipes : pace, offensive rating "
        "et defensive rating."
    ),
    "rosters": (
        "Effectifs et positions des joueurs. "
        "Un joueur transféré peut apparaître plusieurs fois dans une saison."
    ),
}


def find_season_column(frame: pd.DataFrame) -> str | None:
    """
    Recherche la colonne représentant la saison dans une source.
    """
    possible_columns = [
        "SEASON",
        "SEASON_YEAR",
        "Season",
        "season",
    ]

    for column in possible_columns:
        if column in frame.columns:
            return column

    return None


for source_name, frame in consolidated.items():

    # Ignore une source qui n'a pas été extraite ou consolidée
    if frame is None or frame.empty:
        print(f"Source ignorée car vide : {source_name}")
        continue

    season_col = find_season_column(frame)

    if season_col is not None:
        seasons = frame[season_col].dropna().astype(str)

        periode_min = seasons.min() if not seasons.empty else None
        periode_max = seasons.max() if not seasons.empty else None
        nombre_saisons = seasons.nunique()
    else:
        periode_min = None
        periode_max = None
        nombre_saisons = None

    manifest_rows.append({
        "source": f"NBA API - {source_name}",
        "format_original": "API / JSON",
        "fichier_endpoint": source_endpoints.get(
            source_name,
            source_name
        ),
        "lignes": len(frame),
        "colonnes": frame.shape[1],
        "nombre_saisons": nombre_saisons,
        "periode_min": periode_min,
        "periode_max": periode_max,
        "cle_candidate": source_keys.get(
            source_name,
            "À déterminer"
        ),
        "fichier_consolide": (
            f"data/interim/{source_name}_all_seasons.parquet"
        ),
        "observations": source_observations.get(
            source_name,
            (
                "JSON brut et CSV saisonniers conservés ; "
                "Parquet consolidé créé."
            )
        ),
    })


source_manifest = pd.DataFrame(manifest_rows)


# Ordre des colonnes pour faciliter la lecture
manifest_columns = [
    "source",
    "format_original",
    "fichier_endpoint",
    "lignes",
    "colonnes",
    "nombre_saisons",
    "periode_min",
    "periode_max",
    "cle_candidate",
    "fichier_consolide",
    "observations",
]

# Certaines colonnes peuvent ne pas exister dans source_kaggle_summary
for column in manifest_columns:
    if column not in source_manifest.columns:
        source_manifest[column] = None

source_manifest = source_manifest[manifest_columns]


# Enregistrement du manifeste
manifest_path = INTERIM_DIR / "source_manifest.csv"

source_manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8"
)


display(source_manifest)

print("Manifeste enregistré :", manifest_path.resolve())

,source,format_original,fichier_endpoint,lignes,colonnes,nombre_saisons,periode_min,periode_max,cle_candidate,fichier_consolide,observations
0,Kaggle - NBA Players,CSV,all_seasons.csv,12844,22,NaN,1996-97,2022-23,player_name + season,NaN,Chargement réussi ; colonne d'index Unnamed: 0...
1,NBA API - player_bio,API / JSON,LeagueDashPlayerBioStats,12846,25,27.0,1996-97,2022-23,PLAYER_ID + SEASON,data/interim/player_bio_all_seasons.parquet,Profils joueurs et identifiants NBA. JSON brut...
2,NBA API - player_stats_base,API / JSON,LeagueDashPlayerStats - Base,12846,69,27.0,1996-97,2022-23,PLAYER_ID + SEASON,data/interim/player_stats_base_all_seasons.par...,Statistiques traditionnelles par joueur et par...
3,NBA API - player_stats_advanced,API / JSON,LeagueDashPlayerStats - Advanced,12846,81,27.0,1996-97,2022-23,PLAYER_ID + SEASON,data/interim/player_stats_advanced_all_seasons...,"Statistiques avancées par joueur : usage, rati..."
4,NBA API - team_stats_base,API / JSON,LeagueDashTeamStats - Base,802,56,27.0,1996-97,2022-23,TEAM_ID + SEASON,data/interim/team_stats_base_all_seasons.parquet,Statistiques traditionnelles des équipes par s...
5,NBA API - team_stats_advanced,API / JSON,LeagueDashTeamStats - Advanced,802,48,27.0,1996-97,2022-23,TEAM_ID + SEASON,data/interim/team_stats_advanced_all_seasons.p...,"Statistiques avancées des équipes : pace, offe..."
6,NBA API - rosters,API / JSON,CommonTeamRoster,11672,18,27.0,1996-97,2022-23,PLAYER_ID + TEAM_ID + SEASON,data/interim/rosters_all_seasons.parquet,Effectifs et positions des joueurs. Un joueur ...


Manifeste enregistré : C:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\source_manifest.csv


## 9. Vérifications finales de la Phase 1

Ces contrôles ne nettoient pas les données. Ils vérifient seulement que l'extraction a produit des tables utilisables.


In [27]:

source_key_columns = {
    "player_bio": [
        "PLAYER_ID",
        "SEASON"
    ],
    "player_stats_base": [
        "PLAYER_ID",
        "SEASON"
    ],
    "player_stats_advanced": [
        "PLAYER_ID",
        "SEASON"
    ],
    "team_stats_base": [
        "TEAM_ID",
        "SEASON"
    ],
    "team_stats_advanced": [
        "TEAM_ID",
        "SEASON"
    ],
    "rosters": [
        "PLAYER_ID",
        "TeamID",
        "SEASON"
    ],
}


def extraction_checks(
    source_name: str,
    frame: pd.DataFrame,
    key_columns: list[str]
) -> dict:
    """
    Vérifie la présence des données et la qualité minimale
    des clés candidates, sans nettoyer les données.
    """

    if frame is None or frame.empty:
        return {
            "source": source_name,
            "lignes": 0,
            "colonnes": 0,
            "colonnes_cle": " + ".join(key_columns),
            "colonnes_cle_absentes": None,
            "lignes_cle_manquante": None,
            "doublons_cle": None,
            "doublons_exacts": None,
            "nombre_saisons": None,
            "periode_min": None,
            "periode_max": None,
            "statut": "SOURCE VIDE",
        }

    missing_key_columns = [
        column
        for column in key_columns
        if column not in frame.columns
    ]

    if missing_key_columns:
        return {
            "source": source_name,
            "lignes": len(frame),
            "colonnes": frame.shape[1],
            "colonnes_cle": " + ".join(key_columns),
            "colonnes_cle_absentes": ", ".join(
                missing_key_columns
            ),
            "lignes_cle_manquante": None,
            "doublons_cle": None,
            "doublons_exacts": int(
                frame.duplicated().sum()
            ),
            "nombre_saisons": None,
            "periode_min": None,
            "periode_max": None,
            "statut": "COLONNES DE CLÉ ABSENTES",
        }

    season_column = (
        "SEASON"
        if "SEASON" in frame.columns
        else "SEASON_YEAR"
        if "SEASON_YEAR" in frame.columns
        else None
    )

    if season_column is not None:
        seasons = (
            frame[season_column]
            .dropna()
            .astype(str)
        )

        number_of_seasons = seasons.nunique()
        period_min = (
            seasons.min()
            if not seasons.empty
            else None
        )
        period_max = (
            seasons.max()
            if not seasons.empty
            else None
        )
    else:
        number_of_seasons = None
        period_min = None
        period_max = None

    missing_key_rows = int(
        frame[key_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    duplicate_key_rows = int(
        frame.duplicated(
            subset=key_columns,
            keep=False
        ).sum()
    )

    exact_duplicates = int(
        frame.drop(
            columns=["SOURCE_FILE"],
            errors="ignore"
        )
        .duplicated()
        .sum()
    )

    if missing_key_rows == 0 and duplicate_key_rows == 0:
        status = "OK"
    elif missing_key_rows > 0 and duplicate_key_rows > 0:
        status = "CLÉS MANQUANTES ET DOUBLONS"
    elif missing_key_rows > 0:
        status = "CLÉS MANQUANTES"
    else:
        status = "DOUBLONS DE CLÉ À ANALYSER"

    return {
        "source": source_name,
        "lignes": len(frame),
        "colonnes": frame.shape[1],
        "colonnes_cle": " + ".join(key_columns),
        "colonnes_cle_absentes": "",
        "lignes_cle_manquante": missing_key_rows,
        "doublons_cle": duplicate_key_rows,
        "doublons_exacts": exact_duplicates,
        "nombre_saisons": number_of_seasons,
        "periode_min": period_min,
        "periode_max": period_max,
        "statut": status,
    }


checks = []

for source_name, key_columns in source_key_columns.items():

    frame = consolidated.get(
        source_name,
        pd.DataFrame()
    )

    checks.append(
        extraction_checks(
            source_name=source_name,
            frame=frame,
            key_columns=key_columns
        )
    )


extraction_report = pd.DataFrame(checks)


report_columns = [
    "source",
    "lignes",
    "colonnes",
    "colonnes_cle",
    "lignes_cle_manquante",
    "doublons_cle",
    "doublons_exacts",
    "nombre_saisons",
    "periode_min",
    "periode_max",
    "statut",
]

extraction_report = extraction_report[
    report_columns
]


report_path = (
    INTERIM_DIR
    / "extraction_checks.csv"
)

extraction_report.to_csv(
    report_path,
    index=False,
    encoding="utf-8"
)


display(extraction_report)

print(
    "Rapport enregistré :",
    report_path.resolve()
)

,source,lignes,colonnes,colonnes_cle,lignes_cle_manquante,doublons_cle,doublons_exacts,nombre_saisons,periode_min,periode_max,statut
0,player_bio,12846,25,PLAYER_ID + SEASON,0,0,0,27,1996-97,2022-23,OK
1,player_stats_base,12846,69,PLAYER_ID + SEASON,0,0,0,27,1996-97,2022-23,OK
2,player_stats_advanced,12846,81,PLAYER_ID + SEASON,0,0,0,27,1996-97,2022-23,OK
3,team_stats_base,802,56,TEAM_ID + SEASON,0,0,0,27,1996-97,2022-23,OK
4,team_stats_advanced,802,48,TEAM_ID + SEASON,0,0,0,27,1996-97,2022-23,OK
5,rosters,11672,18,PLAYER_ID + TeamID + SEASON,0,0,0,27,1996-97,2022-23,OK


Rapport enregistré : C:\Users\axeln\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\interim\extraction_checks.csv


## Checklist de validation — Phase 1

À cocher après exécution complète :

### Extraction

- [x] CSV principal chargé sans erreur
- [x] Origine et format du CSV documentés
- [x] Test API réussi sur une saison
- [x] Historique `player_bio` extrait
- [x] Historique `player_stats_base` extrait
- [x] Historique `player_stats_advanced` extrait
- [x] Historique `team_stats_base` extrait
- [x] Historique `team_stats_advanced` extrait
- [x] Positions/rosters extraits ou décision de report documentée

### Conservation et reproductibilité

- [x] Source Kaggle brute préservée
- [x] Réponses JSON brutes préservées
- [x] CSV saisonniers préservés
- [x] Tables Parquet consolidées créées
- [x] Journal d'extraction produit
- [x] Manifeste des sources produit


